1. The Dense Layer: The Matrix MultiplierA Dense layer is just a giant affine transformation: $Y = XW^T + b$.Under the hood, PyTorch's nn.Linear just maintains a weight matrix and a bias vector, and computes the dot product.

In [1]:
import torch
import torch.nn as nn

class CustomDenseLayer(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        # 1. Define the Parameters (Weights and Biases)
        # Weights shape: (output_features, input_features)
        self.weights = nn.Parameter(torch.randn(output_features, input_features) * 0.1)
        # Bias shape: (output_features)
        self.bias = nn.Parameter(torch.zeros(output_features))

    def forward(self, x):
        # x shape: (batch_size, input_features)

        # 2. The Math: Y = X * W^T + b
        # We transpose the weights to align the inner dimensions for dot product
        output = torch.matmul(x, self.weights.t()) + self.bias

        return output

x = torch.randn(32, 784) # Batch of 32 images, 784 pixels each
dense = CustomDenseLayer(784, 100)
out = dense(x)
print(f"Dense Output Shape: {out.shape}")

Dense Output Shape: torch.Size([32, 100])


2. The 1D Convolution: The Sliding ScannerWhile 2D Convolutions are for images, 1D Convolutions are incredibly powerful for sequential data. If you are doing NLP tasks (like extracting specific terms from text or analyzing a sentence for sexism classification), a 1D Conv layer acts like an $n$-gram detector, scanning a few words at a time to find local patterns

In [2]:
class TextConv1DLayer(nn.Module):
    def __init__(self, embed_dim, num_filters, kernel_size=3):
        super().__init__()
        # embed_dim: How many numbers represent one word (e.g., 300)
        # num_filters: How many different patterns we want to detect
        # kernel_size: How many words we look at simultaneously (the sliding window)

        self.conv = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=num_filters,
            kernel_size=kernel_size,
            padding=1 # Keeps the sequence length the same
        )

    def forward(self, x):
        # Input x shape: (batch_size, sequence_length, embed_dim)

        # PyTorch Conv1d expects shape: (batch_size, channels, sequence_length)
        # So we must permute the sequence and embedding dimensions
        x = x.permute(0, 2, 1)

        # Apply sliding window convolution
        features = self.conv(x)

        # Permute back to standard NLP shape: (batch_size, sequence_length, num_filters)
        return features.permute(0, 2, 1)

# Batch of 8 sentences, 20 words each, 50-dimensional word embeddings
text_batch = torch.randn(8, 20, 50)
conv1d = TextConv1DLayer(embed_dim=50, num_filters=64, kernel_size=3)
out = conv1d(text_batch)
print(f"Conv1D Output Shape: {out.shape}")

Conv1D Output Shape: torch.Size([8, 20, 64])


3. Self-Attention: The Context EngineThis is the exact core mechanic inside Transformer models. The goal is to look at a sequence of data and dynamically calculate how much every element relates to every other element:$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$Here is how that equation translates line-for-line into code.

In [3]:
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim

        # Linear layers to project input into Queries, Keys, and Values
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, embed_dim)

        # 1. Create Q, K, V matrices
        Q = self.q_proj(x) # What am I looking for?
        K = self.k_proj(x) # What do I contain?
        V = self.v_proj(x) # What is my actual value?

        # 2. Calculate raw attention scores (Dot product of Queries and Keys)
        # We transpose the last two dimensions of K to do matrix multiplication
        # Shape: (batch_size, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1))

        # 3. Scale the scores (Prevents gradients from vanishing)
        scores = scores / math.sqrt(self.embed_dim)

        # 4. Apply Softmax to get probabilities (Attention Weights)
        # Now every row sums to 1.0. It shows exactly where each word is "looking"
        attention_weights = F.softmax(scores, dim=-1)

        # 5. Multiply the weights by the Values
        # Words that got high attention scores will dominate this sum
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

# Batch of 4 sentences, 10 words each, 128-dimensional word embeddings
sequence_batch = torch.randn(4, 10, 128)
attention = ScaledDotProductAttention(embed_dim=128)
out, weights = attention(sequence_batch)

print(f"Attention Output Shape: {out.shape}")
print(f"Attention Weights Shape: {weights.shape}")

Attention Output Shape: torch.Size([4, 10, 128])
Attention Weights Shape: torch.Size([4, 10, 10])


In [4]:
import torch
import torch.nn as nn
import math

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        # 1. Multi-Head Attention (PyTorch has this highly optimized natively)
        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True # Expects shape (batch, seq, feature)
        )

        # 2. Feed-Forward Network (Applied to each position)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_hidden_dim, embed_dim)
        )

        # 3. Normalization Layers
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, embed_dim)

        # --- BLOCK 1: Attention & Add/Norm ---
        # Self-attention: Queries, Keys, and Values are all 'x'
        attn_output, _ = self.attention(query=x, key=x, value=x)

        # Residual connection + LayerNorm
        x = self.norm1(x + self.dropout(attn_output))

        # --- BLOCK 2: Feed-Forward & Add/Norm ---
        ffn_output = self.ffn(x)

        # Residual connection + LayerNorm
        out = self.norm2(x + self.dropout(ffn_output))

        return out

batch_size = 2
sequence_length = 500
embed_dim = 256

# Dummy input
dummy_text_data = torch.randn(batch_size, sequence_length, embed_dim)

# Initialize one Transformer Block
transformer_block = TransformerEncoderBlock(
    embed_dim=256,
    num_heads=8,
    ff_hidden_dim=1024
)

output = transformer_block(dummy_text_data)
print(f"Input Shape:  {dummy_text_data.shape}")
print(f"Output Shape: {output.shape}")
# Notice the shape remains exactly the same! This allows stacking infinite blocks.

Input Shape:  torch.Size([2, 500, 256])
Output Shape: torch.Size([2, 500, 256])
